# Market Risk Engine — Portfolio Pricing Walkthrough

This notebook demonstrates the end-to-end workflow of the Market Risk Engine:

1. **Load market data** (yield curves, vol surfaces, FX rates, commodity curves)
2. **Build a `MarketSnapshot`** for a given valuation date
3. **Parse a portfolio** of mixed instruments from XML
4. **Price every trade** via the `PricingDispatcher`
5. **Analyse results** — NPV table, greeks, PV01 waterfall
6. **Scenario analysis** — parallel rate-shift ladder (+/- 100 bp)
7. **Visualise** yield curves and the EUR/USD vol smile
8. **Bermudan swaption** — Hull-White 1F calibration + trinomial tree demo

---
**Valuation date:** 2024-01-15  
**Portfolio:** `data/sample/portfolio_sample.xml` — 8 trades across rates, FX and commodities

## 1  Imports and path setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure the project root is on the path when running from notebooks/
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)

# ---- Engine imports --------------------------------------------------------
from market_risk_engine.layer1_market_data.loaders import (
    load_yield_curves,
    load_vol_surfaces,
    load_fx_rates,
    load_commodity_curves,
)
from market_risk_engine.layer1_market_data.models import YieldCurve
from market_risk_engine.layer1_market_data.yield_curve import YieldCurveInterpolator
from market_risk_engine.layer2_portfolio.xml_parser import parse_portfolio
from market_risk_engine.layer3_pricing.base import MarketSnapshot, PricingResult
from market_risk_engine.layer3_pricing.dispatcher import PricingDispatcher
from market_risk_engine.common.enums import OptionType, PayReceive

print("Imports OK")

## 2  Valuation date and data paths

In [ ]:
AS_OF = date(2024, 1, 15)

DATA_DIR    = PROJECT_ROOT / "data" / "sample"
PORTFOLIO   = DATA_DIR / "portfolio_sample.xml"

CURVES_CSV  = DATA_DIR / "yield_curves.csv"
VOLS_CSV    = DATA_DIR / "vol_surfaces.csv"
FX_CSV      = DATA_DIR / "fx_rates.csv"
COMM_CSV    = DATA_DIR / "commodity_curves.csv"

print(f"Valuation date : {AS_OF}")
print(f"Portfolio      : {PORTFOLIO}")
print(f"Data directory : {DATA_DIR}")

## 3  Load market data

Each loader reads from a flat CSV keyed on `as_of_date`, filters to the chosen date, and returns a dict of domain-model objects.

In [ ]:
yield_curves     = load_yield_curves(str(CURVES_CSV), AS_OF)
vol_surfaces     = load_vol_surfaces(str(VOLS_CSV),   AS_OF)
fx_rates         = load_fx_rates(str(FX_CSV),         AS_OF)
commodity_curves = load_commodity_curves(str(COMM_CSV), AS_OF)

print("Yield curves loaded :", list(yield_curves.keys()))
print("Vol surfaces loaded :", list(vol_surfaces.keys()))
print("FX rates loaded     :", list(fx_rates.keys()))
print("Commodity curves    :", list(commodity_curves.keys()))

### 3.1  Inspect the USD SOFR curve

In [ ]:
sofr = yield_curves["USD_SOFR"]

df_sofr = pd.DataFrame({
    "Tenor (yr)": sofr.tenors,
    "Zero Rate (%)": [r * 100 for r in sofr.zero_rates],
})
print(f"USD SOFR — {sofr.as_of_date}  |  {sofr.day_count}  |  {sofr.interpolation}")
df_sofr

### 3.2  Visualise yield curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tenors_fine = np.linspace(0.05, 30, 300)

# --- Zero rate curves -------------------------------------------------------
ax = axes[0]
for name, curve in yield_curves.items():
    interp = YieldCurveInterpolator(curve)
    zeros  = [interp.zero_rate(t) * 100 for t in tenors_fine]
    ax.plot(tenors_fine, zeros, label=name, linewidth=2)
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Zero Rate (%)")
ax.set_title(f"Zero-Rate Curves — {AS_OF}")
ax.legend()

# --- Discount factor curves -------------------------------------------------
ax = axes[1]
for name, curve in yield_curves.items():
    interp = YieldCurveInterpolator(curve)
    dfs    = [interp.discount_factor(t) for t in tenors_fine]
    ax.plot(tenors_fine, dfs, label=name, linewidth=2)
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Discount Factor")
ax.set_title(f"Discount Factor Curves — {AS_OF}")
ax.legend()

plt.tight_layout()
plt.show()

### 3.3  Visualise the EUR/USD vol smile

In [ ]:
from market_risk_engine.layer1_market_data.vol_surface import VolSurfaceInterpolator

eurusd_surf  = vol_surfaces["EURUSD"]
eurusd_interp = VolSurfaceInterpolator(eurusd_surf)

strikes_plot = np.linspace(1.00, 1.20, 100)
expiries_plot = [0.0833, 0.25, 0.50]   # 1M, 3M, 6M

fig, ax = plt.subplots(figsize=(9, 4))
for T in expiries_plot:
    vols = [eurusd_interp.get_vol(T, k) * 100 for k in strikes_plot]
    label = f"{int(round(T*12))}M"
    ax.plot(strikes_plot, vols, label=label, linewidth=2)

ax.axvline(fx_rates["EURUSD"].spot, color="grey", linestyle="--", alpha=0.6, label="Spot")
ax.set_xlabel("Strike (EURUSD)")
ax.set_ylabel("Implied Vol (%)")
ax.set_title("EUR/USD Implied Volatility Smile")
ax.legend()
plt.tight_layout()
plt.show()

## 4  Build `MarketSnapshot`

A `MarketSnapshot` is the immutable bundle of all market data for a single valuation date — it is passed read-only to every pricer.

In [ ]:
market = MarketSnapshot(
    as_of_date       = AS_OF,
    yield_curves     = yield_curves,
    vol_surfaces     = vol_surfaces,
    fx_rates         = fx_rates,
    commodity_curves = commodity_curves,
)

print("MarketSnapshot built for", market.as_of_date)
print(f"  {len(market.yield_curves)} yield curves")
print(f"  {len(market.vol_surfaces)} vol surfaces")
print(f"  {len(market.fx_rates)} FX rate objects")
print(f"  {len(market.commodity_curves)} commodity curves")

## 5  Load the portfolio from XML

The XML parser reads trade attributes, populates typed dataclass instances, and groups them into netting sets.

In [ ]:
portfolio = parse_portfolio(str(PORTFOLIO))

print(f"Portfolio id : {portfolio.portfolio_id}")
print(f"As-of date   : {portfolio.as_of_date}")
print(f"Trades       : {len(portfolio.trades)}")
print(f"Netting sets : {list(portfolio.netting_sets.keys())}")

# Quick summary table
rows = []
for t in portfolio.trades:
    rows.append({
        "Trade ID"   : t.trade_id,
        "Type"       : type(t).__name__,
        "Currency"   : getattr(t, "currency", getattr(t, "base_currency", "?")),
        "Notional"   : getattr(t, "notional", getattr(t, "notional_base",
                         getattr(t, "notional_quantity", None))),
        "Netting Set": getattr(t, "netting_set_id", None),
    })

pd.DataFrame(rows)

## 6  Price the full portfolio

`PricingDispatcher` routes each trade to the correct engine and returns a `PricingResult` with NPV and available greeks.

In [ ]:
dispatcher = PricingDispatcher()
results    = dispatcher.price_portfolio(portfolio, market)

# Collect into a DataFrame
type_map = {t.trade_id: type(t).__name__ for t in portfolio.trades}
ns_map   = {t.trade_id: t.netting_set_id for t in portfolio.trades}

rows = []
for r in results:
    rows.append({
        "Trade ID"     : r.trade_id,
        "Type"         : type_map.get(r.trade_id, ""),
        "Netting Set"  : ns_map.get(r.trade_id, ""),
        "Currency"     : r.currency,
        "NPV"          : r.npv,
        "PV01 ($)"     : r.pv01,
        "Vega ($)"     : r.vega,
        "Delta"        : r.delta,
        "Error"        : r.error,
    })

df_results = pd.DataFrame(rows)
df_results

### 6.1  Portfolio summary

In [ ]:
ok = df_results[df_results["Error"].isna()]
print(f"Trades priced successfully : {len(ok)} / {len(df_results)}")
print(f"Total portfolio NPV (mixed CCY) : ${ok['NPV'].sum():>15,.2f}")
print()
print("NPV by netting set:")
print(ok.groupby("Netting Set")["NPV"].sum().apply(lambda x: f"${x:,.2f}").to_string())

### 6.2  NPV bar chart by trade

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in ok["NPV"]]
bars = ax.bar(ok["Trade ID"], ok["NPV"] / 1_000, color=colors, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, ok["NPV"] / 1_000):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + (2 if val >= 0 else -6),
            f"{val:,.1f}k",
            ha="center", va="bottom", fontsize=8)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("NPV ($'000)")
ax.set_title(f"Trade NPV — {AS_OF}")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 7  PV01 waterfall

PV01 (DV01) is the NPV change for a +1 bp parallel shift of the discount curve.  
Only IRS, AmortizingIRS, and FloatFloatSwap currently return a PV01.

In [ ]:
df_pv01 = ok[ok["PV01 ($)"].notna()][["Trade ID", "Type", "NPV", "PV01 ($)"]].copy()

if not df_pv01.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors_pv01 = ["#3498db" if v >= 0 else "#e74c3c" for v in df_pv01["PV01 ($)"]]
    ax.barh(df_pv01["Trade ID"], df_pv01["PV01 ($)"], color=colors_pv01)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("PV01 ($)")
    ax.set_title("PV01 by Trade (+1bp discount curve shift)")
    plt.tight_layout()
    plt.show()
else:
    print("No PV01 data available.")

df_pv01

## 8  Scenario analysis — parallel rate-shift ladder

We reprice the full portfolio under parallel shifts of all USD and EUR yield curves from **-100 bp** to **+100 bp** in 25 bp steps.

In [ ]:
bumps_bp = [-100, -75, -50, -25, 0, 25, 50, 75, 100]

base_npv  = ok["NPV"].sum()
scenario_rows = []

for bump_bp in bumps_bp:
    bump = bump_bp / 10_000   # convert to decimal

    # Build bumped curves
    bumped_curves = {}
    for name, curve in yield_curves.items():
        bumped_curves[name] = YieldCurve(
            currency      = curve.currency,
            curve_name    = curve.curve_name,
            as_of_date    = curve.as_of_date,
            tenors        = curve.tenors,
            zero_rates    = [r + bump for r in curve.zero_rates],
            day_count     = curve.day_count,
            interpolation = curve.interpolation,
        )

    bumped_market = MarketSnapshot(
        as_of_date       = AS_OF,
        yield_curves     = bumped_curves,
        vol_surfaces     = vol_surfaces,
        fx_rates         = fx_rates,
        commodity_curves = commodity_curves,
    )

    bumped_results = dispatcher.price_portfolio(portfolio, bumped_market)
    total_npv = sum(r.npv for r in bumped_results
                    if r.error is None and not (r.npv != r.npv))  # skip NaN

    scenario_rows.append({
        "Shift (bp)": bump_bp,
        "Portfolio NPV ($)": total_npv,
        "NPV Change ($)": total_npv - base_npv,
    })

df_scenarios = pd.DataFrame(scenario_rows)
df_scenarios["Portfolio NPV ($M)"] = df_scenarios["Portfolio NPV ($)"] / 1e6
df_scenarios["NPV Change ($k)"]    = df_scenarios["NPV Change ($)"] / 1e3
df_scenarios[["Shift (bp)", "Portfolio NPV ($M)", "NPV Change ($k)"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Absolute NPV ---
ax = axes[0]
ax.plot(df_scenarios["Shift (bp)"], df_scenarios["Portfolio NPV ($M)"],
        marker="o", linewidth=2, color="#2980b9")
ax.axvline(0, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("Rate Shift (bp)")
ax.set_ylabel("Portfolio NPV ($M)")
ax.set_title("Portfolio NPV vs Parallel Rate Shift")

# --- NPV change ---
ax = axes[1]
chg = df_scenarios["NPV Change ($k)"]
colors_sc = ["#2ecc71" if v >= 0 else "#e74c3c" for v in chg]
ax.bar(df_scenarios["Shift (bp)"], chg, color=colors_sc, width=18)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Rate Shift (bp)")
ax.set_ylabel("NPV Change ($k)")
ax.set_title("NPV Change vs Parallel Rate Shift")

plt.tight_layout()
plt.show()

## 9  Netting-set level analysis

In [ ]:
ns_summary = (
    ok.groupby("Netting Set")
    .agg(
        n_trades   = ("Trade ID", "count"),
        total_npv  = ("NPV", "sum"),
        total_pv01 = ("PV01 ($)", "sum"),
        total_vega = ("Vega ($)", "sum"),
    )
    .reset_index()
)
ns_summary.columns = ["Netting Set", "# Trades", "Net NPV ($)", "Net PV01 ($)", "Net Vega ($)"]

# Pie chart of NPV contribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
npv_vals = ns_summary["Net NPV ($)"]
labels   = ns_summary["Netting Set"]
ax.pie(npv_vals.abs(), labels=labels,
       autopct="%1.1f%%", startangle=90,
       colors=["#3498db", "#2ecc71", "#e67e22"])
ax.set_title("|NPV| Share by Netting Set")

ax = axes[1]
ax.barh(ns_summary["Netting Set"], ns_summary["Net NPV ($)"] / 1_000,
        color=["#3498db", "#2ecc71", "#e67e22"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Net NPV ($k)")
ax.set_title("Net NPV by Netting Set")

plt.tight_layout()
plt.show()

ns_summary

## 10  Bermudan Swaption — Hull-White 1F tree pricer demo

We construct a 5-year Bermudan payer swaption with **annual exercise dates** (4 opportunities) and price it via the Hull-White trinomial tree, calibrated to coterminal European swaption normal vols.

### 10.1  Build a Bachelier (normal) vol surface

We use a flat normal vol of 80 bp across strikes and expiries — realistic for a USD SOFR curve around 4–5%.

In [ ]:
from market_risk_engine.layer1_market_data.models import VolSurface
from market_risk_engine.layer2_portfolio.models import BermudanSwaption
from market_risk_engine.layer3_pricing.bermudan_swaption_pricer import (
    HullWhiteModel,
    calibrate_hull_white,
    BermudanSwaptionPricer,
)
from market_risk_engine.layer3_pricing.swaption_pricer import SwaptionPricer
from market_risk_engine.layer2_portfolio.models import Swaption

# Build a flat 80bp normal vol surface (representative for USD swaptions)
NORMAL_VOL_LEVEL = 0.0080   # 80 bp

berm_vols = VolSurface(
    asset_id    = "USD_NORMAL",
    as_of_date  = AS_OF,
    strikes     = [0.025, 0.030, 0.035, 0.040, 0.045, 0.050, 0.055, 0.060],
    expiries    = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0],
    vols        = np.full((6, 8), NORMAL_VOL_LEVEL),
)

berm_market = MarketSnapshot(
    as_of_date       = AS_OF,
    yield_curves     = yield_curves,
    vol_surfaces     = {"USD_NORMAL": berm_vols},
    fx_rates         = fx_rates,
    commodity_curves = commodity_curves,
)

print(f"Normal vol level : {NORMAL_VOL_LEVEL*10000:.0f} bp")

### 10.2  Define the Bermudan swaption

- **Underlying**: 5y USD SOFR swap, semiannual fixed, starting 2025-01-15
- **Exercise dates**: annually from 2025 to 2028 (4 opportunities)
- **Strike**: at-the-money (equal to the forward par rate)

In [ ]:
# Compute ATM forward swap rate for the 5y swap starting in 1y
sofr_interp   = YieldCurveInterpolator(yield_curves["USD_SOFR"])
t_start       = 1.0   # 1y forward start
t_end         = 6.0   # 5y tenor → 6y from today
atm_strike    = sofr_interp.par_swap_rate(t_start, t_end, "SEMIANNUAL")

print(f"ATM forward par rate ({t_start}y into {int(t_end-t_start)}y) : {atm_strike*100:.4f}%")

EXERCISE_DATES = [
    date(2025, 1, 15),
    date(2026, 1, 15),
    date(2027, 1, 15),
    date(2028, 1, 15),
]
UNDERLYING_START    = date(2025, 1, 15)
UNDERLYING_MATURITY = date(2030, 1, 15)
NOTIONAL            = 10_000_000

berm_trade = BermudanSwaption(
    trade_id            = "BERM_001",
    currency            = "USD",
    notional            = NOTIONAL,
    exercise_dates      = EXERCISE_DATES,
    underlying_start    = UNDERLYING_START,
    underlying_maturity = UNDERLYING_MATURITY,
    strike              = atm_strike,
    option_type         = OptionType.PAYER,
    vol_surface_id      = "USD_NORMAL",
    discount_curve_id   = "USD_SOFR",
    forward_curve_id    = "USD_SOFR",
    payment_frequency   = "SEMIANNUAL",
    day_count           = "ACT365",
    n_tree_steps        = 120,
)

print("\nBermudan swaption:")
print(f"  Strike          : {berm_trade.strike*100:.4f}%")
print(f"  Exercise dates  : {[str(d) for d in berm_trade.exercise_dates]}")
print(f"  Maturity        : {berm_trade.underlying_maturity}")
print(f"  Option type     : {berm_trade.option_type.value}")
print(f"  Tree steps      : {berm_trade.n_tree_steps}")

### 10.3  Calibrate the Hull-White model

HW (a, σ) are calibrated by matching the Bachelier implied vol of each coterminal European swaption to the market vol surface.

In [ ]:
from market_risk_engine.layer1_market_data.vol_surface import VolSurfaceInterpolator

vol_interp = VolSurfaceInterpolator(berm_vols)

a_cal, sigma_cal = calibrate_hull_white(
    exercise_dates    = EXERCISE_DATES,
    swap_maturity     = UNDERLYING_MATURITY,
    strike            = atm_strike,
    payment_frequency = "SEMIANNUAL",
    day_count         = "ACT365",
    opt_type          = OptionType.PAYER,
    today             = AS_OF,
    disc              = sofr_interp,
    vol_interp        = vol_interp,
    notional          = 1.0,
)

print(f"Calibrated Hull-White parameters:")
print(f"  Mean reversion  a     : {a_cal:.6f}")
print(f"  Short-rate vol  sigma : {sigma_cal*10000:.2f} bp")

# Validate: HW-implied vols vs market vols
hw = HullWhiteModel(a_cal, sigma_cal, sofr_interp)

calib_rows = []
for ex_date in EXERCISE_DATES:
    from market_risk_engine.common.date_utils import year_fraction
    from market_risk_engine.layer3_pricing.bermudan_swaption_pricer import _coterminal_data

    entries = _coterminal_data(
        [ex_date], UNDERLYING_MATURITY, atm_strike,
        "SEMIANNUAL", "ACT365", AS_OF, sofr_interp,
    )
    if not entries:
        continue
    e = entries[0]
    sv_mkt = vol_interp.get_vol(e["t0"], atm_strike, forward=e["fsr"])
    sv_hw  = hw.european_swaption_normal_vol(
        e["t0"], e["payment_times"], e["coupon_amounts"],
        1.0, OptionType.PAYER, e["annuity"], e["fsr"], atm_strike,
    )
    calib_rows.append({
        "Exercise Date"  : str(ex_date),
        "Expiry (yr)"    : round(e["t0"], 4),
        "FSR (%)"        : round(e["fsr"] * 100, 4),
        "Mkt Vol (bp)"   : round(sv_mkt * 10000, 2),
        "HW Vol (bp)"    : round(sv_hw  * 10000, 2),
        "Error (bp)"     : round((sv_hw - sv_mkt) * 10000, 3),
    })

pd.DataFrame(calib_rows)

### 10.4  Price the Bermudan swaption

In [ ]:
berm_pricer  = BermudanSwaptionPricer()
berm_result  = berm_pricer.price(berm_trade, berm_market)

print(f"Bermudan swaption NPV : ${berm_result.npv:,.2f}")
print(f"Vega (+1bp sigma bump): ${berm_result.vega:,.2f}")
if berm_result.error:
    print(f"Error: {berm_result.error}")

### 10.5  Compare to coterminal European swaptions

Each coterminal European swaption (Bachelier model) should be cheaper than the Bermudan — the Bermudan always dominates.

In [ ]:
euro_pricer = SwaptionPricer()
euro_rows   = []

for ex_date in EXERCISE_DATES:
    euro = Swaption(
        trade_id            = f"EURO_{ex_date}",
        currency            = "USD",
        notional            = NOTIONAL,
        option_expiry       = ex_date,
        underlying_start    = ex_date,
        underlying_maturity = UNDERLYING_MATURITY,
        strike              = atm_strike,
        option_type         = OptionType.PAYER,
        vol_model           = "bachelier",
        vol_surface_id      = "USD_NORMAL",
        discount_curve_id   = "USD_SOFR",
        forward_curve_id    = "USD_SOFR",
        payment_frequency   = "SEMIANNUAL",
    )
    res = euro_pricer.price(euro, berm_market)
    euro_rows.append({
        "Exercise Date": str(ex_date),
        "European NPV ($)": res.npv,
        "Bermudan NPV ($)": berm_result.npv,
        "Early-Exercise Premium ($)": berm_result.npv - res.npv,
    })

df_comp = pd.DataFrame(euro_rows)
df_comp

### 10.6  Bermudan NPV vs strike (moneyness profile)

In [ ]:
strikes_scan = np.linspace(atm_strike - 0.020, atm_strike + 0.020, 15)
npvs_berm, npvs_euro = [], []

for K in strikes_scan:
    bt = BermudanSwaption(
        trade_id            = "SCAN",
        currency            = "USD",
        notional            = NOTIONAL,
        exercise_dates      = EXERCISE_DATES,
        underlying_start    = UNDERLYING_START,
        underlying_maturity = UNDERLYING_MATURITY,
        strike              = float(K),
        option_type         = OptionType.PAYER,
        vol_surface_id      = "USD_NORMAL",
        discount_curve_id   = "USD_SOFR",
        forward_curve_id    = "USD_SOFR",
        payment_frequency   = "SEMIANNUAL",
        day_count           = "ACT365",
        n_tree_steps        = 80,
    )
    br = berm_pricer.price(bt, berm_market)
    npvs_berm.append(br.npv if not br.error else float("nan"))

    # First European only
    et = Swaption(
        trade_id            = "SCAN_E",
        currency            = "USD",
        notional            = NOTIONAL,
        option_expiry       = EXERCISE_DATES[0],
        underlying_start    = EXERCISE_DATES[0],
        underlying_maturity = UNDERLYING_MATURITY,
        strike              = float(K),
        option_type         = OptionType.PAYER,
        vol_model           = "bachelier",
        vol_surface_id      = "USD_NORMAL",
        discount_curve_id   = "USD_SOFR",
        forward_curve_id    = "USD_SOFR",
        payment_frequency   = "SEMIANNUAL",
    )
    er = euro_pricer.price(et, berm_market)
    npvs_euro.append(er.npv if not er.error else float("nan"))

fig, ax = plt.subplots(figsize=(10, 4))
k_pct = strikes_scan * 100
ax.plot(k_pct, np.array(npvs_berm) / 1000, label="Bermudan (4 exercises)", linewidth=2, color="#2980b9")
ax.plot(k_pct, np.array(npvs_euro) / 1000, label="European (1st exercise)", linewidth=2,
        linestyle="--", color="#e67e22")
ax.fill_between(k_pct,
                np.array(npvs_euro) / 1000,
                np.array(npvs_berm) / 1000,
                alpha=0.15, color="#2980b9", label="Early-exercise premium")
ax.axvline(atm_strike * 100, color="grey", linestyle=":", alpha=0.7, label="ATM")
ax.set_xlabel("Strike (%)")
ax.set_ylabel("NPV ($k)")
ax.set_title("Bermudan vs European Payer Swaption — NPV by Strike")
ax.legend()
plt.tight_layout()
plt.show()

## 11  Summary

| Step | What happened |
|------|---------------|
| **Market data** | Loaded yield curves (USD SOFR, EUR €STR), vol surfaces (USD swaption, EUR/USD, WTI), FX rates and commodity curves from flat CSV files |
| **MarketSnapshot** | Assembled all data into a single immutable object for the valuation date |
| **Portfolio** | Parsed 8 trades (IRS × 2, Swaption, Cap, FX Forward, FX Option, Commodity Swap, Commodity Option) from XML |
| **Pricing** | Dispatched each trade to its pricer; computed NPV + available greeks |
| **Scenarios** | Re-priced under a ±100 bp parallel shift ladder |
| **Bermudan** | Calibrated HW1F to coterminal normal vols, priced via trinomial tree backward induction |

**Key files:**
```
data/sample/yield_curves.csv          → load_yield_curves()
data/sample/vol_surfaces.csv          → load_vol_surfaces()
data/sample/fx_rates.csv              → load_fx_rates()
data/sample/commodity_curves.csv      → load_commodity_curves()
data/sample/portfolio_sample.xml      → parse_portfolio()
data/SOFR data/sofr_curve20260312.csv → load_curve_directory()  (daily format)
```